In [14]:
import pandas as pd
import numpy as np
from sklearn.metrics import mean_absolute_error, mean_squared_error
from sklearn.decomposition import TruncatedSVD
from sklearn.preprocessing import StandardScaler
from sklearn.neighbors import NearestNeighbors
from sklearn.model_selection import train_test_split

data = pd.read_csv('cleaned_I_THINK.csv')

In [15]:
# genres to list
data['genres'] = data['genres'].apply(eval)
# one string
def preprocess_genres(genres):
    return ", ".join(sorted(genres))

data['genres_string'] = data['genres'].apply(preprocess_genres)

In [16]:
# Train and test for user ratings
train_data, test_data = train_test_split(data, test_size=0.2, random_state=42)

In [20]:
# Content-Based Recommendation Model
def recommend_content_based(input_genres, data, top_n=10):
    """
    Recommend movies based on content similarity using genres.
    """
    input_genres_set = set(input_genres.split(", "))

    # Calculate Jaccard similarity
    def genre_similarity(movie_genres):
        return len(input_genres_set.intersection(movie_genres)) / len(input_genres_set.union(movie_genres))

    data['similarity'] = data['genres'].apply(genre_similarity)
    recommendations = data.sort_values(by='similarity', ascending=False).head(top_n)
    return recommendations[['title', 'genres_string', 'user_rating', 'similarity']]

# Collaborative Filtering Model
def collaborative_filtering(data, top_n=10): 
    """
    Collaborative filtering using Truncated SVD for user-item ratings matrix.
    """
    # Create user-item matrix
    user_item_matrix = pd.pivot_table(data, values='user_rating', index='title', columns='release_year', fill_value=0)

    # Normalize
    scaler = StandardScaler()
    user_item_matrix_scaled = scaler.fit_transform(user_item_matrix)

    # Apply SVD
    svd = TruncatedSVD(n_components=20, random_state=42)
    matrix_svd = svd.fit_transform(user_item_matrix_scaled)

    # Train KNN
    model_knn = NearestNeighbors(metric='cosine', algorithm='brute')
    model_knn.fit(matrix_svd)

    # Random movie for recommendation
    movie_indices = np.random.choice(user_item_matrix.index, size=1)
    movie_idx = user_item_matrix.index.get_loc(movie_indices[0])
    distances, indices = model_knn.kneighbors([matrix_svd[movie_idx]], n_neighbors=top_n+1)

    print(f"\nCollaborative Recommendations for '{movie_indices[0]}':")
    recommendations = [user_item_matrix.index[i] for i in indices.flatten()[1:]]
    return recommendations

In [21]:
def evaluate_model(train_data, test_data):
    # MAE AND RMSE
    # actual and predicted ratings
    actual_ratings = test_data['user_rating']
    predicted_ratings = test_data['genres_string'].apply(
    lambda x: train_data[train_data['genres_string'] == x]['user_rating'].mean()
    ).fillna(train_data['user_rating'].mean())

    mae = mean_absolute_error(actual_ratings, predicted_ratings)
    rmse = np.sqrt(mean_squared_error(actual_ratings, predicted_ratings))

    print(f"Evaluation Metrics:")
    print(f"Mean Absolute Error (MAE) is: {mae:}")
    print(f"Root Mean Squared Error (RMSE) is: {rmse:}")

In [25]:
def search_movies(data):
    print("Welcome, choose option:")
    print("1. Search movies by genres (Content-Based)")
    print("2. Recommend movies using collaborative filtering")

    choice = input("Enter your choice (1 or 2): ")

    if choice == "1":
        input_genres = input("Enter genres (comma-separated, e.g., Action, Comedy): ")
        recommendations = recommend_content_based(input_genres, data)
        print("\nContent-Based Recommendations:")
        print(recommendations)
    elif choice == "2":
        recommendations = collaborative_filtering(data)
        print("\nCollaborative Filtering Recommendations:")
        print(recommendations)
    else:
        print("Invalid choice! Please restart and choose 1 or 2.")

if __name__ == "__main__":
    print("### Movie Recommendation System ###\n")
    search_movies(train_data)
    evaluate_model(train_data, test_data)

### Movie Recommendation System ###

Welcome, choose option:
1. Search movies by genres (Content-Based)
2. Recommend movies using collaborative filtering
Invalid choice! Please restart and choose 1 or 2.
Evaluation Metrics:
Mean Absolute Error (MAE) is: 0.08301729315513344
Root Mean Squared Error (RMSE) is: 0.13392403413200982
